# Voix off — notebook Colab (§7.2)

Genere la voix off d'une video a partir de `03_script_tts.txt`, avec clonage de voix (F5-TTS ou Qwen-TTS) et controle qualite automatique.

**Decisions ouvertes (§12), a trancher/ajuster pendant la semaine de test :**
- version exacte et API de Qwen-TTS
- duree ideale de l'extrait de reference par modele
- reduction de bruit : `noisereduce` (leger) ou DeepFilterNet/Demucs (source tres bruitee)
- seuils du controle qualite (cellule 5) et des pauses (cellule 6)

Ce notebook ecrit uniquement dans le dossier de la video et dans
`00_Profil/voix/{nom}/`. Il ne touche a aucun autre `state.json`, et
respecte le contrat du §4.2 (en_cours au debut, termine/echec a la fin,
historique).

## Cellule 0 — Config

In [ ]:
VIDEO_ID = "2026-09-10_v01"  # a renseigner
MODELE = "f5tts"  # "f5tts" ou "qwen_tts"
NOM_VOIX = "voix_principale"
GRAINE = 42  # graine fixe, pour la reproductibilite
NOUVEL_UPLOAD_VOIX = False  # True seulement pour (re)creer la reference de voix

SEUIL_ECART_QC = 0.15  # part de mots divergents tolere avant regeneration (cellule 5, a ajuster)
MAX_TENTATIVES_QC = 3
PAUSE_ENTRE_PHRASES_MS = 250  # a ajuster selon les rapports audio de la semaine de test

## Cellule 1 — Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

RACINE = Path('/content/drive/MyDrive/ChaineYouTube')  # ajuster si raccourci different (§7.5)
DOSSIER_VIDEO = RACINE / 'videos' / VIDEO_ID
assert DOSSIER_VIDEO.is_dir(), f"Dossier video introuvable : {DOSSIER_VIDEO}"
assert (DOSSIER_VIDEO / '03_script_tts.txt').is_file(), "03_script_tts.txt manquant : le CP2 est-il valide ?"

### Contrat state.json (§4.2), en local a ce notebook

Version minimale de `agents/*/scripts/etape.py`, adaptee a Colab (pas d'acces au depot Git, seulement a Drive).

In [ ]:
import json
from datetime import datetime, timezone

def _maintenant_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')

def _lire_state():
    return json.loads((DOSSIER_VIDEO / 'state.json').read_text(encoding='utf-8'))

def _ecrire_state(state):
    chemin = DOSSIER_VIDEO / 'state.json'
    tmp = chemin.with_name(chemin.name + '.tmp')
    tmp.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(tmp, chemin)

def _historique(state, evenement, message):
    state.setdefault('historique', []).append({
        'horodatage': _maintenant_iso(), 'agent': 'colab_voix', 'evenement': evenement, 'message': message,
    })

def etape_commencer():
    state = _lire_state()
    e = state['etapes']['E4_audio']
    # "en_cours" est accepte ici : sur Colab on execute/debug cellule par cellule dans
    # la meme session, donc un statut "en_cours" ne peut venir que d'un run precedent
    # de ce meme notebook interrompu avant d'atteindre etape_terminer/etape_echouer
    # (§4.2 : pas de vraie concurrence a craindre dans ce contexte, contrairement a
    # l'Orchestrateur automatise).
    assert e['statut'] in ('a_venir', 'attente_franco', 'echec', 'en_cours'), f"E4_audio n'est pas pret : {e['statut']}"
    e['tentatives'] = e.get('tentatives', 0) + 1
    e['statut'] = 'en_cours'
    e['debut'] = _maintenant_iso()
    e['fin'] = None
    _historique(state, 'en_cours', f"Run Colab, tentative {e['tentatives']}, modele {MODELE}")
    _ecrire_state(state)

def etape_terminer(sorties, message):
    state = _lire_state()
    e = state['etapes']['E4_audio']
    e['statut'] = 'termine'
    e['fin'] = _maintenant_iso()
    e['sorties'] = sorties
    e['message'] = message
    _historique(state, 'termine', message)
    _ecrire_state(state)

def etape_echouer(message):
    state = _lire_state()
    e = state['etapes']['E4_audio']
    e['statut'] = 'echec'
    e['fin'] = _maintenant_iso()
    e['message'] = message
    _historique(state, 'echec', message)
    _ecrire_state(state)

etape_commencer()

## Cellule 2 — Voix (upload toujours present)

In [ ]:
# Reduction de bruit non branchee ici (§12, a trancher : noisereduce leger
# ou DeepFilterNet/Demucs si la source est tres bruitee). Enregistre pour
# l'instant le wav tel qu'upload.

DOSSIER_VOIX = RACINE / '00_Profil' / 'voix' / NOM_VOIX

def version_courante(dossier_voix):
    if not dossier_voix.is_dir():
        return 0
    versions = [int(p.name[1:]) for p in dossier_voix.iterdir() if p.name.startswith('v') and p.name[1:].isdigit()]
    return max(versions, default=0)

if NOUVEL_UPLOAD_VOIX:
    from google.colab import files

    print("Selectionne les 2 fichiers a uploader : l'audio de reference (.wav) et le texte qu'il contient (.txt).")
    uploades = files.upload()

    fichiers_wav = [n for n in uploades if n.lower().endswith('.wav')]
    fichiers_txt = [n for n in uploades if n.lower().endswith('.txt')]
    assert len(fichiers_wav) == 1, f"Un seul .wav attendu, recu : {fichiers_wav}"
    assert len(fichiers_txt) == 1, f"Un seul .txt attendu, recu : {fichiers_txt}"

    v = version_courante(DOSSIER_VOIX) + 1
    dossier_v = DOSSIER_VOIX / f'v{v}'
    dossier_v.mkdir(parents=True, exist_ok=True)

    ref_wav = dossier_v / 'ref.wav'
    ref_wav.write_bytes(uploades[fichiers_wav[0]])

    ref_txt = uploades[fichiers_txt[0]].decode('utf-8').strip()
    (dossier_v / 'ref.txt').write_text(ref_txt, encoding='utf-8')

    (dossier_v / 'meta.json').write_text(json.dumps({
        'cree_le': _maintenant_iso(),
        'source_wav': fichiers_wav[0],
        'source_txt': fichiers_txt[0],
    }, ensure_ascii=False, indent=2), encoding='utf-8')

    print(f"Voix enregistree : {NOM_VOIX} v{v} ({dossier_v})")
else:
    v = version_courante(DOSSIER_VOIX)
    assert v > 0, f"Aucune voix stockee pour {NOM_VOIX}. Relancer avec NOUVEL_UPLOAD_VOIX=True."
    dossier_v = DOSSIER_VOIX / f'v{v}'
    ref_wav = dossier_v / 'ref.wav'
    ref_txt = (dossier_v / 'ref.txt').read_text(encoding='utf-8')
    print(f"Voix chargee : {NOM_VOIX} v{v}")

## Cellule 3 — Script

In [ ]:
phrases = [l.strip() for l in (DOSSIER_VIDEO / '03_script_tts.txt').read_text(encoding='utf-8').splitlines() if l.strip()]
print(f"{len(phrases)} phrases a synthetiser.")

## Cellule 4 — Synthese (phrase par phrase)

In [ ]:
# pip install -q f5-tts  # a executer une fois par session Colab si pas deja fait

import torch
from f5_tts.api import F5TTS

_f5tts = None

def synthetiser(phrase, graine):
    """Genere l'audio d'une phrase avec F5-TTS, clone sur ref_wav/ref_txt (§7.2, cellule 2).
    Retourne (tableau audio numpy, taux d'echantillonnage)."""
    if MODELE != "f5tts":
        raise NotImplementedError(f"Seul f5tts est branche ici, pas {MODELE} (§7.2, cellule 4).")

    global _f5tts
    if _f5tts is None:
        _f5tts = F5TTS()

    torch.manual_seed(graine)
    wav, sr, _ = _f5tts.infer(
        ref_file=str(ref_wav),
        ref_text=ref_txt,
        gen_text=phrase,
    )
    return wav, sr

clips = [synthetiser(p, GRAINE) for p in phrases]

## Cellule 5 — Controle qualite (Whisper)

In [ ]:
# pip install -q faster-whisper librosa

import numpy as np
import librosa
from faster_whisper import WhisperModel

_modele_qc = None

def _obtenir_modele_qc():
    global _modele_qc
    if _modele_qc is None:
        _modele_qc = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
    return _modele_qc

def transcrire(audio_np, taux):
    """Transcrit un clip avec faster-whisper, pour le controle qualite (§7.2, cellule 5)."""
    if taux != 16000:
        audio_np = librosa.resample(audio_np.astype(np.float32), orig_sr=taux, target_sr=16000)
    segments, _ = _obtenir_modele_qc().transcribe(audio_np, language="en")
    return " ".join(seg.text.strip() for seg in segments)

def ecart_relatif(attendu, obtenu):
    a, o = attendu.lower().split(), obtenu.lower().split()
    if not a:
        return 0.0
    divergents = sum(1 for x, y in zip(a, o) if x != y) + abs(len(a) - len(o))
    return divergents / len(a)

signalements = []
clips_valides = []
for phrase, (audio_np, taux) in zip(phrases, clips):
    ok = False
    for tentative in range(1, MAX_TENTATIVES_QC + 1):
        texte_obtenu = transcrire(audio_np, taux)
        if ecart_relatif(phrase, texte_obtenu) <= SEUIL_ECART_QC:
            ok = True
            break
        audio_np, taux = synthetiser(phrase, GRAINE + tentative)
    if not ok:
        signalements.append({'phrase': phrase, 'derniere_transcription': texte_obtenu})
    clips_valides.append((audio_np, taux))

print(f"{len(signalements)} phrase(s) signalee(s) apres {MAX_TENTATIVES_QC} tentatives.")

## Cellule 6 — Assemblage

In [ ]:
# pip install -q pyloudnorm

import pyloudnorm as pyln

CIBLE_LUFS = -16.0  # loudness cible YouTube/streaming courant, a ajuster (§12, semaine de test)

def normaliser_volume(audio_np, taux):
    """Normalise le volume en LUFS (loudness percue), plus robuste qu'un simple peak/RMS."""
    meter = pyln.Meter(taux)
    loudness = meter.integrated_loudness(audio_np.astype(np.float32))
    if not np.isfinite(loudness):
        return audio_np
    return pyln.normalize.loudness(audio_np.astype(np.float32), loudness, CIBLE_LUFS)

# Tous les clips doivent etre au meme taux d'echantillonnage avant concatenation
taux_final = max(taux for _, taux in clips_valides)
silence = np.zeros(int(taux_final * PAUSE_ENTRE_PHRASES_MS / 1000), dtype=np.float32)

morceaux = []
for i, (audio_np, taux) in enumerate(clips_valides):
    audio_np = audio_np.astype(np.float32)
    if taux != taux_final:
        audio_np = librosa.resample(audio_np, orig_sr=taux, target_sr=taux_final)
    if i > 0:
        morceaux.append(silence)
    morceaux.append(audio_np)

audio_final_np = normaliser_volume(np.concatenate(morceaux), taux_final)
print(f"Audio assemble : {len(audio_final_np) / taux_final:.1f}s, {len(clips_valides)} phrases, taux {taux_final} Hz.")

## Cellule 7 — Timestamps mot par mot

In [ ]:
import soundfile as sf

chemin_wav = DOSSIER_VIDEO / '04_voixoff.wav'
sf.write(chemin_wav, audio_final_np, taux_final)

segments, _ = _obtenir_modele_qc().transcribe(str(chemin_wav), word_timestamps=True, language="en")
mots = [
    {'mot': w.word.strip(), 'debut_s': w.start, 'fin_s': w.end}
    for seg in segments for w in (seg.words or [])
]

with open(DOSSIER_VIDEO / '04_timestamps.json', 'w', encoding='utf-8') as f:
    json.dump(mots, f, ensure_ascii=False, indent=2)

## Cellule 8 — Sorties

In [ ]:
rapport = ['# Rapport audio', '', f"Modele : {MODELE} — Voix : {NOM_VOIX} v{v} — Graine : {GRAINE}", '']
if signalements:
    rapport.append('## Phrases signalees (controle qualite)')
    for s in signalements:
        rapport.append(f"- \"{s['phrase']}\" -> transcrit : \"{s['derniere_transcription']}\"")
else:
    rapport.append('Aucune phrase signalee.')

(DOSSIER_VIDEO / '04_rapport_audio.md').write_text('\n'.join(rapport), encoding='utf-8')

if signalements:
    etape_echouer(f"{len(signalements)} phrase(s) hors seuil apres {MAX_TENTATIVES_QC} tentatives, voir 04_rapport_audio.md")
else:
    etape_terminer(['04_voixoff.wav', '04_timestamps.json', '04_rapport_audio.md'],
                    f"Audio genere, {len(phrases)} phrases, 0 signalement.")

print('Termine. Relance l\'Orchestrateur (ou dis-le a Franco) pour enchainer sur le storyboard/montage.')